# Encoding & Pipelines - Task 4.3: Verify identical predictions

**Week:** 5  
**Source:** `Encoding_Pipelines_PRACTICE.ipynb`

### Task 4.3 — Verify identical predictions
Prove that `log_reg_pipeline` and `pipe_mp` produce IDENTICAL
predictions on `X_test` using `np.array_equal`.


### Prerequisites
Runs the setup and the earlier steps this task depends on (output hidden).

In [ ]:
%%capture
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("Setup complete.")

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df = df.drop(columns=['customerID'])

print("Shape:", df.shape)


df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values in TotalCharges:", df["TotalCharges"].isna().sum())

le = LabelEncoder()
df["Churn_encoded"] = le.fit_transform(df["Churn"])
print({cls: int(code) for cls, code in zip(le.classes_, le.transform(le.classes_))})
print(df[["Churn", "Churn_encoded"]].head())

X = df.drop(columns=["Churn", "Churn_encoded"])
y = df["Churn_encoded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Why split first: frequency/target encoders learn statistics from the data; fitting them on the
# full dataset would leak information from the test set into training (data leakage).
print(X_train.shape, X_test.shape)

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
print("Numeric    :", numeric_features)
print("Categorical:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
preprocessor

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
log_reg_pipeline

log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

pipe_mp = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))
pipe_mp

pipe_mp.fit(X_train, y_train)
print("Pipeline     :", list(log_reg_pipeline.named_steps.keys()))
print("make_pipeline:", list(pipe_mp.named_steps.keys()))

### Solution

In [ ]:
same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
print("Predictions identical:", same)

In [ ]:
try:
    same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
    assert same
    print("Self-check passed — predictions are identical, as expected.")
except NameError:
    print("pipe_mp not fitted yet — complete Task 4.1-4.2 first.")
except AssertionError:
    print("Predictions differ — something in pipe_mp is not built the same way as log_reg_pipeline.")